# Plotly で学ぶ対話的グラフ 入門チュートリアル

このノートブックは、**JupyterLite（ブラウザだけで動く Jupyter 環境）** 上で、
対話的なグラフを描くライブラリ **Plotly** の基礎を学ぶためのチュートリアルです。
Plotly のグラフは、マウスを乗せると値が表示され、ズーム・範囲選択・凡例のクリックによる表示切替・アニメーションができます。

## 対象者
- pandas の DataFrame の基本を理解している方
- matplotlib で静的なグラフを描いたことがある方
- 経済データを対話的に探索したり、Web 上で共有したい方

## このチュートリアルで学ぶこと
1. Plotly とは（plotly.express と graph_objects）
2. plotly.express の基本グラフ（散布図・折れ線・棒・ヒストグラム・箱ひげ・円）
3. 色・サイズ・ホバー・ファセット
4. アニメーション（年ごとの変化）
5. graph_objects による細かな制御（複数トレース、レイアウト、注釈、2 軸）
6. サブプロット
7. 経済データの分析例
8. 保存と共有

## 使い方
- セルを上から順に `Shift + Enter` で実行してください。
- 描いたグラフの右上にはツールバー（ズーム、範囲選択、画像保存など）が表示されます。
- 各章の最後に **練習問題** があります。解答欄に自分で書いてから、折りたたみの解答例を開いて確認しましょう。

---
## 0. 環境準備（JupyterLite 用）

Plotly は JupyterLite に同梱されていないので、PyPI から取得します。**`nbformat` も一緒にインストールしてください。**
Plotly がノートブック上にグラフを表示するときに `nbformat` を必要とし、無いと `fig.show()` がエラーになります。

また、この環境では `plotly.graph_objects` を **先に** import してから `plotly.express` を import します
（`plotly.express` だけを先に import すると、NumPy との相性で読み込みに失敗することがあるためです）。

In [ ]:
# JupyterLite 用のパッケージインストール（初回は数十秒かかります）
try:
    import piplite
    await piplite.install(["plotly", "nbformat", "pandas", "numpy", "statsmodels"])
except ImportError:
    pass

In [ ]:
import plotly.graph_objects as go        # 先に graph_objects を import する
import plotly.express as px
from plotly.subplots import make_subplots
import numpy as np
import pandas as pd
import plotly

print("Plotly バージョン:", plotly.__version__)

### 使用するデータ

乱数で作った **架空の地域経済パネルデータ**（8 地域 × 2015〜2024 年）と、**学生の学習時間と成績のデータ**（200 人）を使います。

In [ ]:
rng = np.random.default_rng(7)

regions = ["北海道", "東北", "関東", "中部", "近畿", "中国", "四国", "九州"]
base_gdp = {"北海道": 20, "東北": 34, "関東": 210, "中部": 90, "近畿": 85, "中国": 30, "四国": 14, "九州": 48}   # 兆円
base_pop = {"北海道": 520, "東北": 850, "関東": 4400, "中部": 2100, "近畿": 2050, "中国": 720, "四国": 360, "九州": 1280}  # 万人

rows = []
for region in regions:
    gdp = base_gdp[region]
    for year in range(2015, 2025):
        growth = rng.normal(1.0, 1.5)
        gdp = gdp * (1 + growth / 100)
        unemployment = rng.normal(2.8, 0.5)
        rows.append({
            "region": region,
            "year": year,
            "gdp": round(gdp, 2),                                   # 域内総生産（兆円）
            "growth": round(growth, 2),                             # 成長率（%）
            "unemployment": round(unemployment, 2),                 # 失業率（%）
            "inflation": round(2.0 - 0.6 * (unemployment - 2.8) + rng.normal(0, 0.3), 2),   # 物価上昇率（%）
            "population": round(base_pop[region] * (1 - 0.003 * (year - 2015)) + rng.normal(0, 5), 1),
        })
econ = pd.DataFrame(rows)
econ["gdp_per_capita"] = (econ["gdp"] / econ["population"] * 10000).round(1)   # 1 人あたり GDP（万円）
print(econ.shape)
print(econ.head())

In [ ]:
faculties = ["経済", "法", "文", "理工"]
n = 200
students = pd.DataFrame({
    "faculty": rng.choice(faculties, size=n),
    "study_hours": rng.gamma(shape=3, scale=1.2, size=n).round(1),
    "part_time": rng.choice(["あり", "なし"], size=n, p=[0.6, 0.4]),
})
students["score"] = (50 + 6 * students["study_hours"] + rng.normal(0, 8, size=n)).clip(0, 100).round(1)
print(students.head())

---
## 1. Plotly とは

Plotly には 2 つの書き方があります。

| モジュール | 特徴 | 使いどころ |
|---|---|---|
| `plotly.express`（`px`） | DataFrame の列名を指定するだけでグラフができる高水準 API | まずはこちら。ほとんどの分析はこれで足りる |
| `plotly.graph_objects`（`go`） | トレース（系列）とレイアウトを 1 つずつ組み立てる低水準 API | 複数系列の細かな制御、2 軸、サブプロット |

どちらも **Figure オブジェクト** を作り、`fig.show()` で表示します。`px` で作った Figure も `go` の方法で後から編集できます。

In [ ]:
# 最初のグラフ：学習時間と成績の散布図（マウスを乗せると値が表示される）
fig = px.scatter(students, x="study_hours", y="score", title="学習時間と成績")
fig.show()

In [ ]:
# Figure の中身：データ（トレース）とレイアウト
print(type(fig))
print("トレースの数:", len(fig.data))
print("トレースの種類:", fig.data[0].type)
print("タイトル:", fig.layout.title.text)

グラフ右上のツールバーで、ズーム（Zoom）、パン（Pan）、範囲選択（Box Select）、リセット（Reset axes）、
PNG として保存（Download plot）ができます。凡例の項目をクリックすると、その系列の表示・非表示を切り替えられます。

### 練習問題 1

1. `students` を使って、横軸 `score`、縦軸 `study_hours` の散布図を描いてください。タイトルは「成績と学習時間」にします。
2. 作った Figure のトレース数と種類を表示してください。

In [ ]:
# 練習問題 1 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 1 の解答例を見る</strong></summary>

```python
# 1
fig = px.scatter(students, x="score", y="study_hours", title="成績と学習時間")
fig.show()

# 2
print(len(fig.data), fig.data[0].type)
```

</details>

---
## 2. plotly.express の基本グラフ

`px` の関数はどれも `px.関数名(データフレーム, x="列", y="列", ...)` という同じ形です。
`labels=` で軸ラベルを日本語にでき、`title=` でタイトルを付けます。

In [ ]:
# 折れ線グラフ：color で地域ごとに線を分ける
fig = px.line(econ, x="year", y="gdp", color="region", markers=True,
              labels={"year": "年", "gdp": "域内総生産（兆円）", "region": "地域"},
              title="地域別 GDP の推移")
fig.show()

In [ ]:
# 棒グラフ：2024 年の地域別 GDP（降順に並べる）
latest = econ[econ["year"] == 2024].sort_values("gdp", ascending=False)
fig = px.bar(latest, x="region", y="gdp", text_auto=".0f",
             labels={"region": "地域", "gdp": "域内総生産（兆円）"},
             title="2024 年の地域別 GDP")
fig.show()

In [ ]:
# グループ化した棒グラフと積み上げ棒グラフ（barmode）
sub = econ[econ["region"].isin(["関東", "中部", "近畿"]) & (econ["year"] >= 2021)]
fig = px.bar(sub, x="year", y="gdp", color="region", barmode="group",
             labels={"year": "年", "gdp": "域内総生産（兆円）", "region": "地域"},
             title="barmode='group'（横に並べる）")
fig.show()

In [ ]:
fig = px.bar(sub, x="year", y="gdp", color="region", barmode="stack",
             labels={"year": "年", "gdp": "域内総生産（兆円）", "region": "地域"},
             title="barmode='stack'（積み上げ）")
fig.show()

横向きの棒グラフは `orientation="h"` にして x と y を入れ替えます。ラベルが長いカテゴリに向いています。

In [ ]:
fig = px.bar(latest.sort_values("gdp"), x="gdp", y="region", orientation="h", text_auto=".0f",
             labels={"region": "地域", "gdp": "域内総生産（兆円）"}, title="2024 年の地域別 GDP（横向き）")
fig.show()

面グラフ（`px.area`）は、地域を積み上げて合計の推移と内訳を同時に見せるのに向いています。

In [ ]:
fig = px.area(econ, x="year", y="gdp", color="region",
              labels={"year": "年", "gdp": "域内総生産（兆円）", "region": "地域"},
              title="地域別 GDP の積み上げ面グラフ")
fig.show()

In [ ]:
# ヒストグラム
fig = px.histogram(students, x="score", nbins=20,
                   labels={"score": "成績", "count": "人数"}, title="成績の分布")
fig.show()

In [ ]:
# 箱ひげ図（points="all" で個々のデータ点も表示）
fig = px.box(students, x="faculty", y="score", color="faculty", points="all",
             labels={"faculty": "学部", "score": "成績"}, title="学部別の成績分布")
fig.show()

In [ ]:
# 円グラフ：2024 年の GDP シェア
fig = px.pie(latest, names="region", values="gdp", title="2024 年の GDP シェア", hole=0.3)
fig.show()

### 練習問題 2

1. `econ` の失業率（`unemployment`）の推移を地域別の折れ線グラフで描いてください（軸ラベルは日本語）。
2. `students` について、`part_time` 別の `study_hours` の箱ひげ図を描いてください。
3. 2024 年の人口（`population`）のシェアを円グラフにしてください。

In [ ]:
# 練習問題 2 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 2 の解答例を見る</strong></summary>

```python
# 1
px.line(econ, x="year", y="unemployment", color="region",
        labels={"year": "年", "unemployment": "失業率（%）", "region": "地域"}, title="地域別の失業率").show()

# 2
px.box(students, x="part_time", y="study_hours", color="part_time",
       labels={"part_time": "アルバイト", "study_hours": "学習時間"}).show()

# 3
px.pie(latest, names="region", values="population", title="2024 年の人口シェア").show()
```

</details>

---
## 3. 色・サイズ・ホバー・ファセット

`px` の関数には、列を視覚属性に対応づける引数が共通で用意されています。

| 引数 | 意味 |
|---|---|
| `color=` | 色（文字列の列 → カテゴリ色、数値の列 → 連続的なカラースケール） |
| `size=` | 点の大きさ（数値の列） |
| `symbol=` | 点の形 |
| `hover_name=`, `hover_data=` | ホバー時に表示する見出しと追加の列 |
| `facet_col=`, `facet_row=` | カテゴリごとに小さなグラフを並べる |
| `category_orders=` | カテゴリの並び順 |

In [ ]:
# 色（カテゴリ）＋大きさ（数値）＋ホバー
fig = px.scatter(latest, x="gdp", y="unemployment", size="population", color="region",
                 hover_name="region", hover_data={"gdp_per_capita": True, "growth": True},
                 size_max=50,
                 labels={"gdp": "域内総生産（兆円）", "unemployment": "失業率（%）", "population": "人口（万人）",
                         "region": "地域", "gdp_per_capita": "1人あたりGDP（万円）", "growth": "成長率（%）"},
                 title="2024 年：GDP・失業率・人口")
fig.show()

In [ ]:
# 数値の列を色にすると連続的なカラースケールになる
fig = px.bar(latest, x="region", y="gdp", color="growth", color_continuous_scale="RdBu",
             color_continuous_midpoint=0,
             labels={"region": "地域", "gdp": "域内総生産（兆円）", "growth": "成長率（%）"},
             title="2024 年の GDP（色 = 成長率）")
fig.show()

In [ ]:
# 点の形（symbol）と、ホバーのテンプレート
fig = px.scatter(students, x="study_hours", y="score", color="faculty", symbol="part_time",
                 labels={"study_hours": "学習時間（時間/日）", "score": "成績", "faculty": "学部", "part_time": "アルバイト"},
                 title="学部・アルバイト別の学習時間と成績")
fig.update_traces(marker=dict(size=9, opacity=0.7))
fig.show()

In [ ]:
# ファセット：学部ごとに小さなグラフを並べる
fig = px.scatter(students, x="study_hours", y="score", color="part_time",
                 facet_col="faculty", facet_col_wrap=2, trendline="ols",
                 labels={"study_hours": "学習時間", "score": "成績", "part_time": "アルバイト"},
                 title="学部別：学習時間と成績（回帰直線付き）", height=600)
fig.show()

### 練習問題 3

1. `latest` を使い、横軸 `population`、縦軸 `gdp`、色 `region`、大きさ `gdp_per_capita`、ホバー見出しに地域名を表示する散布図を描いてください。
2. `econ` の GDP の推移を、`facet_col="region"`（`facet_col_wrap=4`）で地域ごとに並べた折れ線グラフにしてください。

In [ ]:
# 練習問題 3 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 3 の解答例を見る</strong></summary>

```python
# 1
px.scatter(latest, x="population", y="gdp", color="region", size="gdp_per_capita", hover_name="region", size_max=40,
           labels={"population": "人口（万人）", "gdp": "域内総生産（兆円）", "region": "地域"}).show()

# 2
px.line(econ, x="year", y="gdp", facet_col="region", facet_col_wrap=4, height=500,
        labels={"year": "年", "gdp": "GDP（兆円）"}).show()
```

</details>

---
## 4. アニメーション

`animation_frame=` に年などの列を指定すると、再生ボタンとスライダー付きのアニメーションになります。
`animation_group=` で同じ要素（地域）を対応づけ、`range_x` / `range_y` で軸の範囲を固定すると見やすくなります。

In [ ]:
fig = px.scatter(econ, x="gdp_per_capita", y="unemployment", size="population", color="region",
                 animation_frame="year", animation_group="region", hover_name="region", size_max=55,
                 range_x=[0, 600], range_y=[1, 5],
                 labels={"gdp_per_capita": "1 人あたり GDP（万円）", "unemployment": "失業率（%）",
                         "population": "人口（万人）", "region": "地域", "year": "年"},
                 title="年ごとの変化（再生ボタンを押してください）")
fig.show()

In [ ]:
# 棒グラフのアニメーション（年ごとの地域別 GDP）
fig = px.bar(econ, x="region", y="gdp", color="region", animation_frame="year", range_y=[0, 260],
             labels={"region": "地域", "gdp": "域内総生産（兆円）", "year": "年"},
             title="地域別 GDP の推移（アニメーション）")
fig.show()

### 練習問題 4

1. `econ` を使って、横軸 `growth`、縦軸 `inflation`、大きさ `population`、色 `region` の散布図を年ごとのアニメーションにしてください（軸の範囲は `range_x=[-4, 6]`, `range_y=[-1, 5]`）。

In [ ]:
# 練習問題 4 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 4 の解答例を見る</strong></summary>

```python
px.scatter(econ, x="growth", y="inflation", size="population", color="region",
           animation_frame="year", animation_group="region", hover_name="region", size_max=50,
           range_x=[-4, 6], range_y=[-1, 5],
           labels={"growth": "成長率（%）", "inflation": "物価上昇率（%）", "population": "人口（万人）", "region": "地域", "year": "年"}).show()
```

</details>

---
## 5. graph_objects による細かな制御

`go.Figure()` に **トレース**（`go.Scatter`, `go.Bar` など）を `add_trace()` で追加し、
`update_layout()` で **レイアウト**（タイトル、軸、凡例、テンプレート）を設定します。

In [ ]:
kanto = econ[econ["region"] == "関東"]
kinki = econ[econ["region"] == "近畿"]

fig = go.Figure()
fig.add_trace(go.Scatter(x=kanto["year"], y=kanto["gdp"], mode="lines+markers", name="関東"))
fig.add_trace(go.Scatter(x=kinki["year"], y=kinki["gdp"], mode="lines+markers", name="近畿",
                         line=dict(dash="dash")))
fig.update_layout(
    title="関東と近畿の GDP",
    xaxis_title="年",
    yaxis_title="域内総生産（兆円）",
    template="plotly_white",
    legend_title="地域",
    height=400,
)
fig.show()

In [ ]:
# 注釈（annotation）と補助線（hline / vline）
fig = go.Figure(go.Scatter(x=kanto["year"], y=kanto["growth"], mode="lines+markers", name="成長率"))
fig.add_hline(y=0, line_dash="dot", line_color="gray")
fig.add_hline(y=kanto["growth"].mean(), line_dash="dash", line_color="red",
              annotation_text=f"期間平均 {kanto['growth'].mean():.2f}%", annotation_position="top left")
worst = kanto.loc[kanto["growth"].idxmin()]
fig.add_annotation(x=worst["year"], y=worst["growth"], text="最低", showarrow=True, arrowhead=2, ay=40)
fig.update_layout(title="関東の成長率", xaxis_title="年", yaxis_title="成長率（%）", template="plotly_white")
fig.show()

In [ ]:
# 2 軸グラフ：GDP（左軸・棒）と失業率（右軸・線）
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Bar(x=kanto["year"], y=kanto["gdp"], name="GDP（兆円）", opacity=0.6), secondary_y=False)
fig.add_trace(go.Scatter(x=kanto["year"], y=kanto["unemployment"], name="失業率（%）", mode="lines+markers",
                         line=dict(color="crimson")), secondary_y=True)
fig.update_yaxes(title_text="域内総生産（兆円）", secondary_y=False)
fig.update_yaxes(title_text="失業率（%）", secondary_y=True, range=[0, 5])
fig.update_layout(title="関東：GDP と失業率", xaxis_title="年", template="plotly_white")
fig.show()

In [ ]:
# px で作った Figure も同じ方法で編集できる
fig = px.line(econ, x="year", y="unemployment", color="region", labels={"year": "年", "unemployment": "失業率（%）", "region": "地域"})
fig.update_traces(line=dict(width=1.5))
fig.update_layout(title="失業率の推移（テンプレートと凡例位置を変更）", template="plotly_dark",
                  legend=dict(orientation="h", y=-0.2))
fig.show()

### 練習問題 5

1. `go.Figure` を使って、中部と九州の失業率の推移を折れ線（マーカー付き）で重ねてください。タイトル・軸ラベルは日本語にします。
2. そのグラフに、全地域・全期間の失業率の平均を赤い破線の水平線として追加してください。

In [ ]:
# 練習問題 5 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 5 の解答例を見る</strong></summary>

```python
chubu = econ[econ["region"] == "中部"]
kyushu = econ[econ["region"] == "九州"]
fig = go.Figure()
fig.add_trace(go.Scatter(x=chubu["year"], y=chubu["unemployment"], mode="lines+markers", name="中部"))
fig.add_trace(go.Scatter(x=kyushu["year"], y=kyushu["unemployment"], mode="lines+markers", name="九州"))
fig.add_hline(y=econ["unemployment"].mean(), line_dash="dash", line_color="red", annotation_text="全体平均")
fig.update_layout(title="中部と九州の失業率", xaxis_title="年", yaxis_title="失業率（%）", template="plotly_white")
fig.show()
```

</details>

---
## 6. サブプロット

`make_subplots(rows=, cols=)` で複数のグラフ領域を作り、`add_trace(..., row=, col=)` でそれぞれに描きます。

In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=("2024 年の GDP", "2024 年の失業率"))
fig.add_trace(go.Bar(x=latest["region"], y=latest["gdp"], name="GDP"), row=1, col=1)
fig.add_trace(go.Bar(x=latest["region"], y=latest["unemployment"], name="失業率", marker_color="orange"), row=1, col=2)
fig.update_yaxes(title_text="兆円", row=1, col=1)
fig.update_yaxes(title_text="%", row=1, col=2)
fig.update_layout(height=400, showlegend=False, template="plotly_white")
fig.show()

In [ ]:
# 2 行 × 2 列：4 地域の成長率
picks = ["北海道", "関東", "近畿", "九州"]
fig = make_subplots(rows=2, cols=2, subplot_titles=picks, shared_yaxes=True)
for i, region in enumerate(picks):
    d = econ[econ["region"] == region]
    fig.add_trace(go.Bar(x=d["year"], y=d["growth"], name=region), row=i // 2 + 1, col=i % 2 + 1)
fig.update_layout(height=550, title="地域別の成長率（%）", showlegend=False, template="plotly_white")
fig.show()

### 練習問題 6

1. 1 行 2 列のサブプロットに、左に `students` の成績のヒストグラム（`go.Histogram`）、右に学習時間のヒストグラムを描いてください。

In [ ]:
# 練習問題 6 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 6 の解答例を見る</strong></summary>

```python
fig = make_subplots(rows=1, cols=2, subplot_titles=("成績", "学習時間"))
fig.add_trace(go.Histogram(x=students["score"], nbinsx=20, name="成績"), row=1, col=1)
fig.add_trace(go.Histogram(x=students["study_hours"], nbinsx=20, name="学習時間", marker_color="seagreen"), row=1, col=2)
fig.update_layout(height=400, showlegend=False, template="plotly_white")
fig.show()
```

</details>

---
## 7. 経済データの分析例

マクロ経済のデータ分析でよく描くグラフを、Plotly で作ってみます。

In [ ]:
# 例 1：フィリップス曲線（失業率と物価上昇率の関係）に回帰直線を重ねる
fig = px.scatter(econ, x="unemployment", y="inflation", color="region", trendline="ols", trendline_scope="overall",
                 hover_data=["year"],
                 labels={"unemployment": "失業率（%）", "inflation": "物価上昇率（%）", "region": "地域"},
                 title="フィリップス曲線（全地域・全期間、黒線は全体の回帰直線）")
fig.update_traces(line=dict(color="black"), selector=dict(mode="lines"))
fig.show()

In [ ]:
# 例 2：2015 年を 100 とした GDP 指数（レンジスライダー付き）
first = econ[econ["year"] == 2015][["region", "gdp"]].rename(columns={"gdp": "gdp_2015"})
indexed = econ.merge(first, on="region")
indexed["index"] = (indexed["gdp"] / indexed["gdp_2015"] * 100).round(1)

fig = px.line(indexed, x="year", y="index", color="region", markers=True,
              labels={"year": "年", "index": "GDP 指数（2015 年 = 100）", "region": "地域"},
              title="地域別 GDP 指数（下のスライダーで期間を絞れます）")
fig.add_hline(y=100, line_dash="dot", line_color="gray")
fig.update_xaxes(rangeslider_visible=True)
fig.show()

In [ ]:
# 例 3：ヒートマップ（地域 × 年の成長率）
pivot = econ.pivot(index="region", columns="year", values="growth")
fig = px.imshow(pivot, color_continuous_scale="RdBu", zmin=-4, zmax=4, aspect="auto", text_auto=".1f",
                labels={"x": "年", "y": "地域", "color": "成長率（%）"}, title="地域 × 年の成長率")
fig.show()

### 練習問題 7

1. 横軸 1 人あたり GDP（`gdp_per_capita`）、縦軸失業率の散布図（2024 年）に、地域名のテキストラベルを表示してください（`text="region"`, `fig.update_traces(textposition="top center")`）。
2. `pivot`（地域 × 年）を失業率で作り直し、ヒートマップにしてください。

In [ ]:
# 練習問題 7 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 7 の解答例を見る</strong></summary>

```python
# 1
fig = px.scatter(latest, x="gdp_per_capita", y="unemployment", text="region",
                 labels={"gdp_per_capita": "1 人あたり GDP（万円）", "unemployment": "失業率（%）"}, title="2024 年")
fig.update_traces(textposition="top center", marker=dict(size=10))
fig.show()

# 2
pivot_u = econ.pivot(index="region", columns="year", values="unemployment")
px.imshow(pivot_u, color_continuous_scale="Oranges", aspect="auto", text_auto=".1f",
          labels={"x": "年", "y": "地域", "color": "失業率（%）"}).show()
```

</details>

---
## 8. 保存と共有

`fig.write_html()` で **対話機能付きの HTML ファイル** として保存できます。
`include_plotlyjs="cdn"` にするとファイルが小さくなります（表示にはインターネット接続が必要）。
保存したファイルは左のファイルブラウザに現れ、右クリック → Download で手元に保存できます。

PNG などの画像への変換（`fig.write_image()`）には追加ライブラリ `kaleido` が必要で、JupyterLite では使えません。
画像が欲しいときは、グラフ右上のツールバーにあるカメラのアイコンから保存してください。

In [ ]:
fig = px.line(econ, x="year", y="gdp", color="region", title="地域別 GDP の推移")
fig.write_html("gdp_trend.html", include_plotlyjs="cdn")
print("gdp_trend.html を保存しました")

# グラフの定義は JSON として取り出せる
print(fig.to_json()[:200], "...")

---
## まとめ

| トピック | 主な書き方 |
|---|---|
| 基本 | `px.scatter / line / bar / histogram / box / pie(df, x=, y=, color=, labels=, title=)` → `fig.show()` |
| 見た目 | `color=`, `size=`, `symbol=`, `hover_name=`, `hover_data=`, `color_continuous_scale=`, `text_auto=` |
| 分割 | `facet_col=`, `facet_row=`, `facet_col_wrap=` |
| アニメーション | `animation_frame=`, `animation_group=`, `range_x=`, `range_y=` |
| graph_objects | `go.Figure()`, `add_trace(go.Scatter / go.Bar / go.Histogram)`, `update_layout()`, `update_traces()` |
| 注釈・補助線 | `add_annotation()`, `add_hline()`, `add_vline()` |
| 2 軸・サブプロット | `make_subplots(specs=[[{"secondary_y": True}]])`, `make_subplots(rows=, cols=)`, `row=, col=` |
| 分析向け | `trendline="ols"`, `rangeslider_visible=True`, `px.imshow(pivot)` |
| 保存 | `fig.write_html("x.html", include_plotlyjs="cdn")`, `fig.to_json()` |

## 次のステップ
- `python/altair/altair_beginner_tutorial.ipynb` — 宣言的な文法で対話グラフを描く Altair
- `python/ipywidgets/ipywidgets_beginner_tutorial.ipynb` — スライダーなどの部品と組み合わせる
- `python/statsmodels/statsmodels_tutorial.ipynb` — 回帰直線の背後にある回帰分析

---
## 総合演習：地域経済ダッシュボード

`econ` を使って、次の 2 つの Figure を作ってください。

1. **2 行 2 列のサブプロット**：
   - (1,1) 地域別 GDP の推移（折れ線、地域ごとに 1 本）
   - (1,2) 2024 年の地域別失業率（棒、降順）
   - (2,1) 2024 年の 1 人あたり GDP と失業率の散布図（点にホバーで地域名）
   - (2,2) 関東の GDP（棒・左軸）と物価上昇率（線・右軸）の 2 軸グラフ
   全体に日本語のタイトルを付け、`template="plotly_white"` にします。
2. 1 人あたり GDP（横軸）と失業率（縦軸）の散布図を、年ごとのアニメーションにし（点の大きさは人口）、HTML ファイルとして保存する。

In [ ]:
# 総合演習の解答欄：ここにコードを書いてください

### 総合演習の解答例

In [ ]:
# 1. 2 行 2 列のサブプロット
latest = econ[econ["year"] == 2024].sort_values("unemployment", ascending=False)
kanto = econ[econ["region"] == "関東"]

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=("地域別 GDP の推移", "2024 年の失業率", "2024 年：1 人あたり GDP と失業率", "関東：GDP と物価上昇率"),
    specs=[[{}, {}], [{}, {"secondary_y": True}]],
)
for region in regions:
    d = econ[econ["region"] == region]
    fig.add_trace(go.Scatter(x=d["year"], y=d["gdp"], mode="lines", name=region), row=1, col=1)
fig.add_trace(go.Bar(x=latest["region"], y=latest["unemployment"], name="失業率", marker_color="orange", showlegend=False), row=1, col=2)
fig.add_trace(go.Scatter(x=latest["gdp_per_capita"], y=latest["unemployment"], mode="markers", text=latest["region"],
                         hovertemplate="%{text}<br>1人あたりGDP %{x} 万円<br>失業率 %{y}%", marker=dict(size=12),
                         name="地域", showlegend=False), row=2, col=1)
fig.add_trace(go.Bar(x=kanto["year"], y=kanto["gdp"], name="GDP", opacity=0.5, showlegend=False), row=2, col=2, secondary_y=False)
fig.add_trace(go.Scatter(x=kanto["year"], y=kanto["inflation"], mode="lines+markers", name="物価上昇率", line=dict(color="crimson"),
                         showlegend=False), row=2, col=2, secondary_y=True)
fig.update_yaxes(title_text="兆円", row=1, col=1)
fig.update_yaxes(title_text="%", row=1, col=2)
fig.update_xaxes(title_text="1 人あたり GDP（万円）", row=2, col=1)
fig.update_yaxes(title_text="失業率（%）", row=2, col=1)
fig.update_yaxes(title_text="GDP（兆円）", row=2, col=2, secondary_y=False)
fig.update_yaxes(title_text="物価上昇率（%）", row=2, col=2, secondary_y=True)
fig.update_layout(height=750, title="地域経済ダッシュボード（架空データ）", template="plotly_white", legend_title="地域")
fig.show()

# 2. アニメーション + HTML 保存
fig2 = px.scatter(econ, x="gdp_per_capita", y="unemployment", size="population", color="region",
                  animation_frame="year", animation_group="region", hover_name="region", size_max=55,
                  range_x=[0, 600], range_y=[1, 5],
                  labels={"gdp_per_capita": "1 人あたり GDP（万円）", "unemployment": "失業率（%）", "population": "人口（万人）",
                          "region": "地域", "year": "年"},
                  title="1 人あたり GDP と失業率の推移")
fig2.write_html("dashboard_animation.html", include_plotlyjs="cdn")
print("dashboard_animation.html を保存しました")
fig2.show()

お疲れさまでした！ `px` で素早く描き、`go` と `make_subplots` で仕上げる、という Plotly の使い方を身につけました。
HTML として保存すれば、対話機能付きのグラフをそのまま Web ページやレポートに埋め込めます。